In [31]:
import networkx as nx
import random 
import math

G = nx.Graph()

edges = [
    (1,2,1000),
    (1,3,1500),
    (1,8,2400),
    (2,3,600),
    (2,4,1200),
    (3,4,800),
    (4,5,700),
    (5,6,900),
    (6,7,800),
    (7,8,1000),
    (3,6,1600),
    (2,7,1700)
]

for u,v,w in edges:
    G.add_edge(u,v, weight=w)

nodes = list(G.nodes())

In [32]:


# ----------------------------
# Modulation table
# ----------------------------
modulation_formats = [
    {"name": "BPSK", "reach": 8000, "spectral_eff": 1},
    {"name": "QPSK", "reach": 4000, "spectral_eff": 2},
    {"name": "8QAM", "reach": 2000, "spectral_eff": 3},
    {"name": "16QAM", "reach": 1000, "spectral_eff": 4},
]

SLOT_WIDTH = 12.5  # GHz
GUARD_BAND = 1     # 1 slot

# ----------------------------
# Function to choose modulation
# ----------------------------
def choose_modulation(distance):
    # Choose highest spectral efficiency that satisfies reach
    for mod in reversed(modulation_formats):
        if distance <= mod["reach"]:
            return mod
    return modulation_formats[0]  # fallback to BPSK


# ----------------------------
# Generate 1000 requests
# ----------------------------
requests = []

for i in range(1000):
    src, dst = random.sample(nodes, 2)
    bandwidth = random.choice([50, 100, 200])  # Gbps

    # Shortest path distance
    path = nx.shortest_path(G, src, dst, weight='weight')
    distance = nx.shortest_path_length(G, src, dst, weight='weight')

    # Choose modulation
    mod = choose_modulation(distance)

    # Slot capacity per slot
    slot_capacity = mod["spectral_eff"] * SLOT_WIDTH

    # Required slots
    slots = math.ceil(bandwidth / slot_capacity) + GUARD_BAND

    requests.append({
        "id": i,
        "source": src,
        "destination": dst,
        "bandwidth": bandwidth,
        "path": path,
        "distance": distance,
        "modulation": mod["name"],
        "slots_required": slots
    })

# Print first 5 requests
for r in requests[0:]:
    print(r)

{'id': 0, 'source': 7, 'destination': 6, 'bandwidth': 50, 'path': [7, 6], 'distance': 800, 'modulation': '16QAM', 'slots_required': 2}
{'id': 1, 'source': 4, 'destination': 1, 'bandwidth': 100, 'path': [4, 2, 1], 'distance': 2200, 'modulation': 'QPSK', 'slots_required': 5}
{'id': 2, 'source': 2, 'destination': 4, 'bandwidth': 200, 'path': [2, 4], 'distance': 1200, 'modulation': '8QAM', 'slots_required': 7}
{'id': 3, 'source': 6, 'destination': 5, 'bandwidth': 100, 'path': [6, 5], 'distance': 900, 'modulation': '16QAM', 'slots_required': 3}
{'id': 4, 'source': 8, 'destination': 1, 'bandwidth': 100, 'path': [8, 1], 'distance': 2400, 'modulation': 'QPSK', 'slots_required': 5}
{'id': 5, 'source': 4, 'destination': 6, 'bandwidth': 200, 'path': [4, 5, 6], 'distance': 1600, 'modulation': '8QAM', 'slots_required': 7}
{'id': 6, 'source': 6, 'destination': 5, 'bandwidth': 100, 'path': [6, 5], 'distance': 900, 'modulation': '16QAM', 'slots_required': 3}
{'id': 7, 'source': 8, 'destination': 2, 'b

In [33]:
# Initialize spectrum map for all links
F = 720  # your size
all_links = [frozenset(edge) for edge in G.edges()]
spectrum_map = {link: [-1] * F for link in all_links}

In [34]:
import random

blocked = 0
allocated = 0
blocked_requests = []

for req in requests:
    path = req["path"]
    slots_needed = req["slots_required"]

    # Convert path nodes → link list
    path_links = [frozenset({path[i], path[i+1]}) for i in range(len(path)-1)]

    # All possible starting indices
    possible_starts = list(range(F - slots_needed + 1))
    random.shuffle(possible_starts)  # Random allocation

    allocated_flag = False

    for start in possible_starts:
        end = start + slots_needed

        # Check continuity + contiguity
        feasible = True
        for link in path_links:
            if any(spectrum_map[link][s] != -1 for s in range(start, end)):
                feasible = False
                break

        if feasible:
            # Allocate on ALL links
            for link in path_links:
                for s in range(start, end):
                    spectrum_map[link][s] = req["id"]

                    

            allocated += 1
            allocated_flag = True
            break

    if not allocated_flag:
        blocked += 1
        blocked_requests.append(req)

print("Allocated:", allocated)
print("Blocked:", blocked)
print("Blocking Probability:", blocked / len(requests))

Allocated: 801
Blocked: 199
Blocking Probability: 0.199


In [35]:
for link in spectrum_map:
    print(f"\nLink {set(link)}:")
    print(spectrum_map[link][:100])  # show first 100 slots


Link {1, 2}:
[-1, -1, -1, -1, -1, -1, 432, 432, 432, 432, 432, 432, 432, 432, 432, -1, -1, -1, -1, -1, 478, 478, 478, 478, 478, 297, 297, 297, 297, 297, 297, 297, 297, 297, -1, 267, 267, -1, -1, 740, 740, 740, -1, -1, -1, 542, 542, 542, -1, -1, 419, 419, 449, 449, 449, 449, 449, -1, 302, 302, -1, -1, -1, -1, 184, 184, 184, 103, 103, 103, -1, 400, 400, 400, 400, 400, -1, -1, -1, 224, 224, 224, 224, 224, 224, 224, 224, 224, 601, 601, 601, -1, -1, -1, -1, -1, -1, -1, -1, 852]

Link {1, 3}:
[-1, -1, 101, 101, 101, 101, 101, 101, 101, -1, -1, -1, -1, -1, -1, -1, -1, 935, 935, 935, -1, -1, -1, -1, -1, -1, 950, 950, 950, 950, 950, 950, 950, 475, 475, 475, 475, 475, 475, 475, -1, 938, 938, 938, 938, -1, -1, 112, 112, 112, 112, 112, 112, 112, 112, 112, -1, -1, -1, 120, 120, 120, -1, 476, 476, 476, 476, 476, 476, 476, 476, 476, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 62, 62, 62, 211, 211, 211, 211, 211, 927, 927, 927, 927, 927, -1, -1]

Link {8, 1}:
[-1, 531, 531, 531, 531, 531, -1,

In [36]:
import csv

filename = "requests_basic.csv"

with open(filename, mode="w", newline="") as file:
    writer = csv.writer(file)

    # Header
    writer.writerow(["source", "destination", "required_slots"])

    # Write only required fields
    for req in requests:
        writer.writerow([
            req["source"],
            req["destination"],
            req["slots_required"]
        ])

print("CSV file created:", filename)

CSV file created: requests_basic.csv


In [37]:
from collections import Counter

# Extract required_slots
slot_values = [req["slots_required"] for req in requests]

# Count frequency
slot_counts = Counter(slot_values)

# Print sorted results
print("Unique slot sizes:", sorted(slot_counts.keys()))
print("\nFrequency of each slot size:")
for slot_size in sorted(slot_counts):
    print(f"{slot_size} slots → {slot_counts[slot_size]} requests")

Unique slot sizes: [2, 3, 4, 5, 7, 9]

Frequency of each slot size:
2 slots → 84 requests
3 slots → 322 requests
4 slots → 104 requests
5 slots → 219 requests
7 slots → 124 requests
9 slots → 147 requests


In [38]:
def reallocate_request_exact(req_id, spectrum_map, requests_dict, F):
    info = requests_dict[req_id]
    path_links = info["path"]
    old_start, old_end = info["slots"]
    k = old_end - old_start + 1

    # ---- Temporarily free old slots ----
    for link in path_links:
        for s in range(old_start, old_end + 1):
            spectrum_map[link][s] = -1

    # ---- Compute common free slots ----
    common = [0]*F
    for i in range(F):
        for link in path_links:
            if spectrum_map[link][i] != -1:
                common[i] = 1
                break

    # ---- Find free blocks ----
    blocks = []
    i = 0
    while i < F:
        if common[i] == 0:
            start = i
            while i < F and common[i] == 0:
                i += 1
            end = i - 1
            size = end - start + 1
            blocks.append((start, size))
        else:
            i += 1

    # ---- Exact-Fit selection ----
    exact_blocks = [b for b in blocks if b[1] == k]

    if exact_blocks:
        new_start = exact_blocks[0][0]
    else:
        candidates = [b for b in blocks if b[1] >= k]
        if not candidates:
            # Restore old allocation
            for link in path_links:
                for s in range(old_start, old_end + 1):
                    spectrum_map[link][s] = req_id
            return False
        new_start = min(candidates, key=lambda x: x[1])[0]

    new_end = new_start + k - 1

    # ---- Allocate new position ----
    for link in path_links:
        for s in range(new_start, new_end + 1):
            spectrum_map[link][s] = req_id

    # ---- Update dictionary ----
    requests_dict[req_id]["slots"] = (new_start, new_end)

    return True

In [39]:
def defragment_exact(spectrum_map, requests_dict, F):
    moved = 0

    # Sort by slot size (small first)
    sorted_requests = sorted(
        requests_dict.keys(),
        key=lambda r: requests_dict[r]["slots"][1] - requests_dict[r]["slots"][0]
    )

    for req_id in sorted_requests:
        if reallocate_request_exact(req_id, spectrum_map, requests_dict, F):
            moved += 1

    print("Lightpaths moved:", moved)

In [40]:
requests_dict = {}

for link, slots in spectrum_map.items():
    for i, req_id in enumerate(slots):
        if req_id != -1:
            if req_id not in requests_dict:
                requests_dict[req_id] = {
                    "slots": [i, i],
                    "path": set([link])
                }
            else:
                requests_dict[req_id]["slots"][1] = i
                requests_dict[req_id]["path"].add(link)

# Convert structures properly
for req_id in requests_dict:
    start, end = requests_dict[req_id]["slots"]
    requests_dict[req_id]["slots"] = (start, end)
    requests_dict[req_id]["path"] = list(requests_dict[req_id]["path"])

In [41]:
defragment_exact(spectrum_map, requests_dict, F)

Lightpaths moved: 801


In [42]:
def allocate_blocked_exact(req, spectrum_map, requests_dict, F):
    path = req["path"]
    k = req["slots_required"]
    req_id = req["id"]

    path_links = [frozenset({path[i], path[i+1]})
                  for i in range(len(path)-1)]

    # Compute common free slots
    common = [0]*F
    for i in range(F):
        for link in path_links:
            if spectrum_map[link][i] != -1:
                common[i] = 1
                break

    # Find free blocks
    blocks = []
    i = 0
    while i < F:
        if common[i] == 0:
            start = i
            while i < F and common[i] == 0:
                i += 1
            size = i - start
            blocks.append((start, size))
        else:
            i += 1

    # Exact-Fit rule
    exact_blocks = [b for b in blocks if b[1] == k]

    if exact_blocks:
        start = exact_blocks[0][0]
    else:
        candidates = [b for b in blocks if b[1] >= k]
        if not candidates:
            return False
        start = min(candidates, key=lambda x: x[1])[0]

    end = start + k - 1

    # Allocate
    for link in path_links:
        for s in range(start, end+1):
            spectrum_map[link][s] = req_id

    # Add to allocated dictionary
    requests_dict[req_id] = {
        "source": req["source"],
        "destination": req["destination"],
        "slots": (start, end),
        "path": path_links
    }

    return True

In [43]:
newly_allocated = 0
still_blocked = []

for req in blocked_requests:
    success = allocate_blocked_exact(req, spectrum_map, requests_dict, F)
    if success:
        newly_allocated += 1
    else:
        still_blocked.append(req)

print("Newly Allocated After Defragmentation:", newly_allocated)
print("Still Blocked:", len(still_blocked))

Newly Allocated After Defragmentation: 25
Still Blocked: 174


In [44]:
iteration = 0

while True:
    iteration += 1
    print(f"\nIteration {iteration}")

    # Step 1: Defragment
    defragment_exact(spectrum_map, requests_dict, F)

    # Step 2: Try allocating blocked
    newly_allocated = 0
    new_blocked = []

    for req in blocked_requests:
        success = allocate_blocked_exact(req, spectrum_map, requests_dict, F)
        if success:
            newly_allocated += 1
        else:
            new_blocked.append(req)

    print("New allocations this round:", newly_allocated)

    # Stop condition
    if newly_allocated == 0:
        print("No further improvement possible.")
        break

    blocked_requests = new_blocked


Iteration 1
Lightpaths moved: 826
New allocations this round: 17

Iteration 2
Lightpaths moved: 830
New allocations this round: 2

Iteration 3
Lightpaths moved: 831
New allocations this round: 1

Iteration 4
Lightpaths moved: 832
New allocations this round: 0
No further improvement possible.


In [45]:
from collections import Counter

slot_values = [req["slots_required"] for req in requests]
slot_counts = Counter(slot_values)

In [46]:
total_requests = sum(slot_counts.values())

slot_proportions = {
    k: slot_counts[k] / total_requests
    for k in sorted(slot_counts)
}

In [47]:
print(slot_proportions)

{2: 0.084, 3: 0.322, 4: 0.104, 5: 0.219, 7: 0.124, 9: 0.147}


In [48]:
region_sizes = {
    k: int(round(slot_proportions[k] * F))
    for k in slot_proportions
}

In [49]:
print(region_sizes)

{2: 60, 3: 232, 4: 75, 5: 158, 7: 89, 9: 106}


In [50]:
difference = F - sum(region_sizes.values())

# Add remaining slots to largest demand class
largest_class = max(region_sizes, key=region_sizes.get)
region_sizes[largest_class] += difference

In [51]:
sum(region_sizes.values()) == 720

True

In [52]:
region_boundaries = {}

current_start = 0

for k in sorted(region_sizes):
    size = region_sizes[k]
    start = current_start
    end = current_start + size - 1
    region_boundaries[k] = (start, end)
    current_start = end + 1

In [53]:
print(region_boundaries)

{2: (0, 59), 3: (60, 291), 4: (292, 366), 5: (367, 524), 7: (525, 613), 9: (614, 719)}


In [54]:
def is_inside_region(start, end, region_start, region_end):
    return start >= region_start and end <= region_end

In [55]:
def migrate_to_region(req_id, spectrum_map, requests_dict, 
                      region_boundaries, F):

    info = requests_dict[req_id]
    path_links = info["path"]
    old_start, old_end = info["slots"]
    k = old_end - old_start + 1

    region_start, region_end = region_boundaries[k]

    # If already inside region → nothing to do
    if is_inside_region(old_start, old_end, 
                        region_start, region_end):
        return False

    # ---- Temporarily free old slots ----
    for link in path_links:
        for s in range(old_start, old_end + 1):
            spectrum_map[link][s] = -1

    # ---- Compute common free slots inside region ----
    common = [0] * F

    for i in range(region_start, region_end + 1):
        for link in path_links:
            if spectrum_map[link][i] != -1:
                common[i] = 1
                break

    # ---- Find contiguous free blocks inside region ----
    blocks = []
    i = region_start

    while i <= region_end:
        if common[i] == 0:
            start = i
            while i <= region_end and common[i] == 0:
                i += 1
            size = i - start
            blocks.append((start, size))
        else:
            i += 1

    # ---- Exact-Fit inside region ----
    exact_blocks = [b for b in blocks if b[1] == k]

    if exact_blocks:
        new_start = exact_blocks[0][0]
    else:
        candidates = [b for b in blocks if b[1] >= k]
        if not candidates:
            # Restore old allocation
            for link in path_links:
                for s in range(old_start, old_end + 1):
                    spectrum_map[link][s] = req_id
            return False

        new_start = min(candidates, key=lambda x: x[1])[0]

    new_end = new_start + k - 1

    # ---- Allocate new region placement ----
    for link in path_links:
        for s in range(new_start, new_end + 1):
            spectrum_map[link][s] = req_id

    # Update dictionary
    requests_dict[req_id]["slots"] = (new_start, new_end)

    return True

In [56]:
def soft_region_migration(spectrum_map, requests_dict, 
                          region_boundaries, F):

    moved = 0

    for req_id in requests_dict.keys():
        if migrate_to_region(req_id, spectrum_map, 
                             requests_dict, 
                             region_boundaries, F):
            moved += 1

    print("Moved into preferred region:", moved)

In [57]:
soft_region_migration(
    spectrum_map,
    requests_dict,
    region_boundaries,
    F
)

Moved into preferred region: 190


In [58]:
misplaced = 0

for req_id, info in requests_dict.items():
    start, end = info["slots"]
    k = end - start + 1
    region_start, region_end = region_boundaries[k]

    if not (start >= region_start and end <= region_end):
        misplaced += 1

print("Total allocated:", len(requests_dict))
print("Misplaced lightpaths:", misplaced)

Total allocated: 832
Misplaced lightpaths: 488
